<a href="https://colab.research.google.com/github/lhgiangg/NLP/blob/main/Lab6_truyenkieu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import re
import numpy as np
from collections import Counter
from google.colab import files

# Cấu hình thiết bị (Sử dụng GPU nếu có trên Colab)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang chạy trên thiết bị: {device}")

# Mở hộp thoại để bạn chọn file truyen kieu.txt từ máy tính
print("Vui lòng upload file truyen_kieu.txt:")
uploaded = files.upload()

# Giả sử file của bạn được đặt tên là 'truyen_kieu.txt'
# Nếu tên file của bạn có dấu cách 'truyen kieu.txt', hãy đổi lại biến filename cho khớp
filename = list(uploaded.keys())[0]
with open(filename, 'r', encoding='utf-8') as f:
    text_data = f.readlines()

Đang chạy trên thiết bị: cpu
Vui lòng upload file truyen_kieu.txt:


Saving lab_2_truyen_kieu.txt to lab_2_truyen_kieu.txt


In [ ]:
# Tiền xử lý: Chuyển chữ thường và loại bỏ dấu câu (chỉ giữ lại chữ cái và khoảng trắng)
def preprocess_text(lines):
    corpus = []
    for line in lines:
        line = line.lower()
        line = re.sub(r'[^a-zàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ\s]', '', line)
        words = line.split()
        if len(words) > 0:
            corpus.append(words)
    return corpus

corpus = preprocess_text(text_data)

# Xây dựng từ vựng (Vocabulary)
all_words = [word for sentence in corpus for word in sentence]
word_counts = Counter(all_words)
vocab = list(word_counts.keys())
vocab_size = len(vocab)

# Tạo từ điển ánh xạ: Từ -> Index và Index -> Từ
word2idx = {w: idx for (idx, w) in enumerate(vocab)}
idx2word = {idx: w for (idx, w) in enumerate(vocab)}

print(f"Kích thước tập từ vựng: {vocab_size} từ (âm tiết) duy nhất.")

Kích thước tập từ vựng: 2393 từ (âm tiết) duy nhất.


In [ ]:
# 4. Tạo dữ liệu Positive & Negative (Negative Sampling)
WINDOW_SIZE = 2
NUM_NEG_SAMPLES = 5 # Với mỗi 1 mẫu positive, tạo ra 5 mẫu negative

def generate_sgns_data(corpus, word2idx, window_size, num_neg_samples):
    data_pairs = []
    labels = []
    vocab_sz = len(word2idx)

    for sentence in corpus:
        indices = [word2idx[word] for word in sentence]
        for i, target in enumerate(indices):
            context_start = max(0, i - window_size)
            context_end = min(len(indices), i + window_size + 1)

            for j in range(context_start, context_end):
                if i != j:
                    context = indices[j]
                    # --- Mẫu Positive (Nhãn 1) ---
                    data_pairs.append((target, context))
                    labels.append(1.0)

                    # --- Mẫu Negative (Nhãn 0) ---
                    # Bốc ngẫu nhiên num_neg_samples từ trong từ điển làm nhiễu
                    for _ in range(num_neg_samples):
                        neg_context = np.random.randint(0, vocab_sz)
                        data_pairs.append((target, neg_context))
                        labels.append(0.0)

    return data_pairs, labels

print("Đang tạo dữ liệu Positive và Negative...")
data_pairs, labels = generate_sgns_data(corpus, word2idx, WINDOW_SIZE, NUM_NEG_SAMPLES)
print(f"Tổng số cặp huấn luyện: {len(data_pairs)} (bao gồm cả Negative)")

x_target = torch.tensor([pair[0] for pair in data_pairs], dtype=torch.long)
x_context = torch.tensor([pair[1] for pair in data_pairs], dtype=torch.long)
y_labels = torch.tensor(labels, dtype=torch.float32) # Nhãn dùng float cho hàm loss

Đang tạo dữ liệu Positive và Negative...
Tổng số cặp huấn luyện: 430092 (bao gồm cả Negative)


In [ ]:
# 5. Khởi tạo Mô hình Hồi quy Logistic (SGNS Model)
EMBEDDING_DIM = 100

class SGNS_LogisticRegression(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super(SGNS_LogisticRegression, self).__init__()
        # Ta có 2 bảng tra cứu: một cho từ đóng vai trò Trung tâm, một cho từ đóng vai trò Ngữ cảnh/Nhiễu
        self.target_embeddings = nn.Embedding(vocab_size, embed_dim)
        self.context_embeddings = nn.Embedding(vocab_size, embed_dim)

    def forward(self, target_word, context_word):
        # Lấy vector [batch_size, embed_dim]
        target_embeds = self.target_embeddings(target_word)
        context_embeds = self.context_embeddings(context_word)

        # Logistic Regression: Tính tích vô hướng (Dot Product) giữa 2 vector
        # score = w^T * x
        score = torch.sum(target_embeds * context_embeds, dim=1)

        # Trả về score (Hàm Loss bên dưới sẽ tự động áp dụng Sigmoid)
        return score

model = SGNS_LogisticRegression(vocab_size, EMBEDDING_DIM).to(device)

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader # Import here

# 6. Huấn luyện (Training Loop) với Binary Cross Entropy
LEARNING_RATE = 0.005
EPOCHS = 20 # Có Negative Sampling nên hội tụ nhanh hơn, không cần train quá nhiều epoch
BATCH_SIZE = 1024

# BCEWithLogitsLoss tự động áp dụng hàm Sigmoid lên output của mô hình (tốt cho phân loại nhị phân Logistic Regression)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

dataset = TensorDataset(x_target, x_context, y_labels)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Bắt đầu huấn luyện Logistic Regression (Negative Sampling)...")
for epoch in range(EPOCHS):
    total_loss = 0
    for batch_target, batch_context, batch_y in dataloader:
        batch_target = batch_target.to(device)
        batch_context = batch_context.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        # Dự đoán (Tích vô hướng)
        predictions = model(batch_target, batch_context)

        # Tính Loss giữa dự đoán và nhãn thực tế (1 hoặc 0)
        loss = criterion(predictions, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch: {epoch + 1}/{EPOCHS} | Loss: {total_loss/len(dataloader):.4f}")

print("Huấn luyện hoàn tất!")

Bắt đầu huấn luyện Logistic Regression (Negative Sampling)...
Epoch: 1/20 | Loss: 3.5238
Epoch: 5/20 | Loss: 0.3918
Epoch: 10/20 | Loss: 0.1457
Epoch: 15/20 | Loss: 0.1086
Epoch: 20/20 | Loss: 0.0969
Huấn luyện hoàn tất!


In [ ]:
import torch.nn.functional as F

# 7. Trích xuất và kiểm tra Word Embeddings
# Lấy trọng số của Target Embeddings làm kết quả cuối cùng
word_embeddings = model.target_embeddings.weight.data.cpu()

def get_similar_words(word, top_n=5):
    if word not in word2idx:
        return "Từ này không có trong từ vựng."
    word_idx = word2idx[word]
    word_vec = word_embeddings[word_idx].unsqueeze(0)
    cos_sim = F.cosine_similarity(word_vec, word_embeddings)
    top_indices = torch.topk(cos_sim, top_n + 1).indices.tolist()[1:]
    return [idx2word[idx] for idx in top_indices]

test_words = ['kiều', 'xuân', 'tình', 'trăng']
print("\n--- Kiểm tra kết quả Word Embedding ---")
for w in test_words:
    if w in word2idx:
        print(f"Các từ gần ngữ cảnh nhất với '{w}': {get_similar_words(w)}")


--- Kiểm tra kết quả Word Embedding ---
Các từ gần ngữ cảnh nhất với 'kiều': ['dư', 'lại', 'người', 'gặp', 'nẻo']
Các từ gần ngữ cảnh nhất với 'xuân': ['lâu', 'huyên', 'chay', 'thoắt', 'dông']
Các từ gần ngữ cảnh nhất với 'tình': ['nhớ', 'ngành', 'khi', 'lại', 'nghĩ']
Các từ gần ngữ cảnh nhất với 'trăng': ['liều', 'chồng', 'gọi', 'cốt', 'nay']
